# Jurimetria com a API DataJud (Colab)

Notebook para consulta e analise jurimetrica com a API publica DataJud.

In [ ]:
!pip -q install pandas requests matplotlib

In [ ]:
# Preencha sua chave no formato APIKey ...
API_KEY = "APIKey COLE_SUA_CHAVE_AQUI"
TRIBUNAL = "tjmg"  # ex: tjmg, tjsp, tjrj
CLASSE_CODIGO = 12729
NUMERO_PROCESSO = ""  # opcional
QUANTIDADE = 700

In [ ]:
from typing import Any

import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display

URL_TEMPLATE = "https://api-publica.datajud.cnj.jus.br/api_publica_{tribunal}/_search"


def normalize_api_key(raw_key: str) -> str:
    key = (raw_key or "").strip().strip('"').strip("'")
    if key.lower().startswith("authorization:"):
        key = key.split(":", 1)[1].strip()
    if not key:
        return ""
    if key.startswith("APIKey ") or key.startswith("Bearer "):
        return key
    return f"APIKey {key}"


def normalize_numero_processo(raw_numero: str) -> str:
    numero = (raw_numero or "").strip()
    somente_digitos = "".join(ch for ch in numero if ch.isdigit())
    return somente_digitos or numero


def parse_assuntos(assuntos: Any) -> list[str]:
    if not isinstance(assuntos, list):
        return []
    out = []
    for a in assuntos:
        if isinstance(a, dict):
            out.append(str(a.get("nome", "")))
        else:
            out.append(str(a))
    return out


def fetch_hits(api_key: str, tribunal: str, classe_codigo: int, quantidade: int, numero_processo: str = "") -> list[dict[str, Any]]:
    tribunal = (tribunal or "tjmg").strip().lower()
    url = URL_TEMPLATE.format(tribunal=tribunal)

    numero_limpo = normalize_numero_processo(numero_processo)
    if numero_limpo:
        query = {"match": {"numeroProcesso": numero_limpo}}
    else:
        query = {"match": {"classe.codigo": int(classe_codigo)}}

    payload = {
        "size": int(quantidade),
        "_source": [
            "numeroProcesso",
            "classe.nome",
            "dataAjuizamento",
            "dataHoraUltimaAtualizacao",
            "orgaoJulgador.nome",
            "orgaoJulgador.codigoMunicipioIBGE",
            "assuntos",
        ],
        "query": query,
        "sort": [{"dataAjuizamento": {"order": "desc"}}],
    }

    headers = {
        "Authorization": normalize_api_key(api_key),
        "Content-Type": "application/json",
    }

    resp = requests.post(url, headers=headers, json=payload, timeout=120)
    resp.raise_for_status()
    return resp.json().get("hits", {}).get("hits", [])


def hits_to_dataframe(hits: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for hit in hits:
        src = hit.get("_source", {}) if isinstance(hit, dict) else {}
        classe = src.get("classe", {}) if isinstance(src.get("classe"), dict) else {}
        orgao = src.get("orgaoJulgador", {}) if isinstance(src.get("orgaoJulgador"), dict) else {}

        rows.append({
            "numero_processo": src.get("numeroProcesso"),
            "classe": classe.get("nome"),
            "data_ajuizamento": src.get("dataAjuizamento"),
            "ultima_atualizacao": src.get("dataHoraUltimaAtualizacao"),
            "orgao_julgador": orgao.get("nome"),
            "municipio": orgao.get("codigoMunicipioIBGE"),
            "assuntos": parse_assuntos(src.get("assuntos", [])),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df["data_ajuizamento"] = pd.to_datetime(df["data_ajuizamento"], utc=True, errors="coerce").dt.tz_convert("America/Sao_Paulo")
    df["ultima_atualizacao"] = pd.to_datetime(df["ultima_atualizacao"], utc=True, errors="coerce").dt.tz_convert("America/Sao_Paulo")
    return df

In [ ]:
hits = fetch_hits(
    api_key=API_KEY,
    tribunal=TRIBUNAL,
    classe_codigo=CLASSE_CODIGO,
    quantidade=QUANTIDADE,
    numero_processo=NUMERO_PROCESSO,
)

df = hits_to_dataframe(hits)
print(f"Registros retornados: {len(df)}")

if len(df) > 0:
    total_assuntos_unicos = df["assuntos"].explode().dropna().astype(str).nunique()
    total_orgaos = df["orgao_julgador"].nunique()
    print(f"Assuntos unicos: {total_assuntos_unicos}")
    print(f"Orgaos julgadores: {total_orgaos}")
    display(df.head(20))
else:
    print("Sem dados para os filtros informados.")

In [ ]:
if len(df) > 0:
    top_orgaos = (
        df.groupby(["municipio", "orgao_julgador"]) ["numero_processo"]
        .count()
        .sort_values(ascending=False)
        .head(20)
        .rename("quantidade")
        .reset_index()
    )
    display(top_orgaos)
else:
    print("Top orgaos indisponivel sem dados.")

In [ ]:
if len(df) > 0:
    horas = df["data_ajuizamento"].dropna().dt.hour.value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(horas.index, horas.values, color="#4E79A7")
    ax.set_title("Horario de ajuizamento")
    ax.set_xlabel("Hora")
    ax.set_ylabel("Quantidade")
    ax.grid(axis="y", alpha=0.3)
    plt.show()

    em_expediente = horas[(horas.index >= 9) & (horas.index <= 19)].sum()
    fora_expediente = horas.sum() - em_expediente

    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.pie(
        [em_expediente, fora_expediente],
        labels=["9h-19h", "Fora"],
        autopct="%1.1f%%",
        colors=["#A0CBE8", "#FFBE7D"],
        startangle=30,
    )
    ax.set_title("Expediente x fora")
    plt.show()

In [ ]:
if len(df) > 0:
    mensal = (
        df["data_ajuizamento"].dropna()
        .dt.tz_convert("UTC").dt.tz_localize(None)
        .to_frame(name="data")
        .set_index("data")
        .resample("ME")
        .size()
        .sort_index()
        .tail(12)
    )

    if len(mensal) == 0:
        print("Sem dados mensais.")
    else:
        fig, ax = plt.subplots(figsize=(10, 4))
        labels = [x.strftime("%m/%Y") for x in mensal.index]
        ax.bar(labels, mensal.values, color="#59A14F")
        ax.set_title("Ajuizamentos mensais (ultimos 12 meses disponiveis)")
        ax.set_xlabel("Mes")
        ax.set_ylabel("Quantidade")
        ax.grid(axis="y", alpha=0.3)
        plt.xticks(rotation=45, ha="right")
        plt.show()